# Data Cleaning – leesmateriaal.csv

In dit notebook wordt de leescatalogus (`leesmateriaal.csv`) stap voor stap opgeschoond.

Stappen:
1. Data inladen en bekijken
2. Lege kolom verwijderen en kolomnamen consistent maken
3. Categorische waarden consistent maken
4. Trefwoorden, onderwerp en thema's opschonen
5. Numerieke kolommen opschonen
6. Dubbele `Materiaal_id`'s corrigeren
7. Doelgroep combineren en overbodige kolommen verwijderen
8. Gegevensstatus en publicatiejaar corrigeren
9. Type materiaal en genre opschonen
10. Beschikbaar formaat en auteurs opschonen
11. Resultaat controleren en opslaan

## 1) Data inladen en bekijken

In [ ]:
import pandas as pd

df = pd.read_csv("../datasets/leesmateriaal.csv")

print("Shape:", df.shape)
df.head()

Shape: (283, 33)


,Titel,Auteur,Korte omschrijving,Unnamed: 3,Type materiaal,Niveau,Themas_boek,materiaal_id,genre,onderwerp,...,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,interne_batchcode,administratieve_prioriteit,doelgroep_extra,genre_kopie,Gemiddelde beoordeling
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,NaN,boek,3F,WOII;onderduik,LM-0001,fictie,WOII,...,VERHALEN,Educatief Media,2025,PDF,CAT-00001,B1,Hoog,NaN,fictie,3.4
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,NaN,Boek,3F,spanning;stalking,LM-0002,fictie,spanning,...,verhalen,Leeswereld,2005,Digitaal,CAT-00002,B2,hoog,NT2,fictie,2.5
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",NaN,Boek,2F,liefde;relaties,LM-0003,fictie,liefde,...,verhalen,NaN,2025,Digitaal,CAT-00003,B2,Midden,NT2,fictie,3.8
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,NaN,Boek,3F,ontmoeting;verbondenheid,LM-0004,fictie,ontmoeting,...,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004,batch-2,Laag,NT2,fictie,3.9
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,NaN,Boek,2F,geheimen;schuld,LM-0005,fictie,geheimen,...,verhalen,Uitgeverij Noord,NaN,online,CAT-00005,B3,Laag,MBO/NT2,fictie,3.1


In [2]:
# Datatypes, aantal ontbrekende en unieke waarden per kolom
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "ontbrekend": df.isna().sum(),
    "uniek": df.nunique(),
})

,dtype,ontbrekend,uniek
Titel,str,0,277
Auteur,str,7,231
Korte omschrijving,str,0,282
Unnamed: 3,float64,283,0
Type materiaal,str,1,10
Niveau,str,6,4
Themas_boek,str,134,133
materiaal_id,str,0,279
genre,str,26,3
onderwerp,str,181,47


## 2) Lege kolom verwijderen en kolomnamen consistent maken
De kolomnamen in het bestand zijn niet consistent (`Type materiaal`, `materiaal_id`, `Gemiddelde beoordeling`).
We vervangen spaties door `_` en beginnen elke naam met een hoofdletter, bijvoorbeeld `Type_materiaal` en `Materiaal_id`.

In [3]:
# de unnamed kolom verwijderen, want hij heeft nergens waarden
df = df.drop(columns=['Unnamed: 3'])

# consistente kolomnamen: spaties -> '_' en eerste letter een hoofdletter
df.columns = [kolom.strip().replace(" ", "_") for kolom in df.columns]
df.columns = [kolom[0].upper() + kolom[1:] for kolom in df.columns]

df.columns.tolist()

['Titel',
 'Auteur',
 'Korte_omschrijving',
 'Type_materiaal',
 'Niveau',
 'Themas_boek',
 'Materiaal_id',
 'Genre',
 'Onderwerp',
 'Trefwoorden',
 'Taal',
 'Aantal_paginas',
 'Geschatte_leestijd_min',
 'Bron_url',
 'Doelgroep',
 'Gegevensstatus',
 'Woordenaantal_omschrijving',
 'Gem_woordlengte',
 'Gem_zinslengte_woorden',
 'Aandeel_lange_woorden_pct',
 'Aantal_zinnen',
 'Leescomplexiteit_score',
 'Interessecluster',
 'Uitgever',
 'Publicatiejaar',
 'Beschikbaar_formaat',
 'Cataloguscode',
 'Interne_batchcode',
 'Administratieve_prioriteit',
 'Doelgroep_extra',
 'Genre_kopie',
 'Gemiddelde_beoordeling']

## 3) Categorische waarden consistent maken
Dezelfde waarde staat vaak op verschillende manieren in de data (bijv. `Boek` / `boek`, `NL` / `NED` / `Nederlands`).
We halen spaties weg, maken alles lowercase en zetten spellingsvarianten om naar één waarde.

In [4]:
# Varianten vóór het opschonen
for kolom in ["Type_materiaal", "Administratieve_prioriteit", "Beschikbaar_formaat",
              "Interessecluster", "Doelgroep", "Doelgroep_extra", "Taal", "Interne_batchcode"]:
    print(kolom, "->", df[kolom].dropna().unique().tolist())

Type_materiaal -> ['boek', 'Boek', 'tijdschrift', 'krant (papier)', 'digitale krant', 'Boek - thriller', 'Boek - roman', 'online artikel', 'dichtbundel', 'blogpost']
Administratieve_prioriteit -> ['Hoog', 'hoog', 'Midden', 'Laag']
Beschikbaar_formaat -> ['PDF', 'Digitaal', 'online', 'papier', 'Papier', 'Online']
Interessecluster -> ['VERHALEN', 'verhalen', 'vrije tijd', 'maatschappij']
Doelgroep -> ['MBO', 'MBO/NT2', 'NT2', 'mbo', 'mbo / nt2']
Doelgroep_extra -> ['NT2', 'MBO/NT2', 'MBO']
Taal -> ['Nederlands', 'NED', 'NL', 'nederlands']
Interne_batchcode -> ['B1', 'B2', 'batch-2', 'B3']


In [5]:
# Data Consistent maken:

df["Type_materiaal"] = (
    df["Type_materiaal"]
    .str.strip()
    .str.lower()
)

df["Administratieve_prioriteit"] = (
    df["Administratieve_prioriteit"]
    .str.strip()
    .str.lower()
)

df["Beschikbaar_formaat"] = (
    df["Beschikbaar_formaat"]
    .str.strip()
    .str.lower()
)

df["Interessecluster"] = (
    df["Interessecluster"]
    .str.strip()
    .str.lower()
)

df["Doelgroep"] = (
    df["Doelgroep"]
    .str.replace(" ","", regex=False)
    .str.lower()
)

df["Taal"] = df["Taal"].replace({
    "NED": "Nederlands",
    "NL": "Nederlands",
    "nederlands": "Nederlands"
})

df["Interne_batchcode"] = df["Interne_batchcode"].replace({
    "batch-2": "B2",
})

df["Doelgroep_extra"] = (
    df["Doelgroep_extra"]
    .str.replace(" ","", regex=False)
    .str.lower()
)

df["Auteur"] = df["Auteur"].replace({
    "Noami Alderman": "Naomi Alderman",
    "H. Mulisch": "Harry Mulisch",
    "Annejet vd Zijl": "Annejet van der Zijl"
})

## 4) Trefwoorden, onderwerp en thema's opschonen
- Scheidingstekens (`,` `/` `|` `;`) worden overal `;`
- Bronnen (URL's met `.nl` of `.com`) worden verwijderd
- Lege waarden worden consistent `NA`

In [6]:
# ook bronnen weggehaald
df["Trefwoorden"] = (
    df["Trefwoorden"]
    .str.replace(r"\s*[,/|;]\s*", ";", regex=True)
    .str.strip()
    .str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True)
    .replace(["", "nan"], pd.NA)
)

# Bronnen weggehaald bij Onderwerp
df["Onderwerp"] = df["Onderwerp"].str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True).replace(["", "nan"], pd.NA)

# Bronnen weggehaald, constistente naamgeving (er was ; , / |), en consistente null
df["Themas_boek"] = (
    df["Themas_boek"]
    .str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True, case=False)
    .str.replace(r"\s*[,/|;]\s*", ";", regex=True)
    .str.strip(" ;")
    .replace(["", "nan"], pd.NA)
)

df[["Trefwoorden", "Onderwerp", "Themas_boek"]].head(10)

,Trefwoorden,Onderwerp,Themas_boek
0,WOII;onderduik,WOII,WOII;onderduik
1,spanning;stalking,spanning,spanning;stalking
2,liefde;relaties,liefde,liefde;relaties
3,ontmoeting;verbondenheid,ontmoeting,ontmoeting;verbondenheid
4,geheimen;schuld,geheimen,geheimen;schuld
5,familie;humor,familie,familie;humor
6,reizen;herinneringen,reizen,reizen;herinneringen
7,ontmoeting;hoop,ontmoeting,ontmoeting;hoop
8,ontmoeting;inzicht,ontmoeting,ontmoeting;inzicht
9,oorlog;verlies,oorlog,oorlog;verlies


## 5) Numerieke kolommen opschonen

In [7]:
# Aantal pagina's opschonen: - en 0 waarden weg, consistente NaN en int in plaats van float.
df["Aantal_paginas"] = df["Aantal_paginas"].mask(
    df["Aantal_paginas"] <= 0
).astype("Int64")

# bij Aandeel_lange_woorden_pct de '%' verwijderd en een float gemaakt
df["Aandeel_lange_woorden_pct"] = (
    df["Aandeel_lange_woorden_pct"].str.replace("%", "", regex=False)
)
df["Aandeel_lange_woorden_pct"] = df.Aandeel_lange_woorden_pct.astype(float)

df[["Aantal_paginas", "Aandeel_lange_woorden_pct"]].describe()

,Aantal_paginas,Aandeel_lange_woorden_pct
count,255.0,283.000000
mean,184.996078,42.430035
std,112.073215,21.840599
min,1.0,0.000000
25%,118.5,28.600000
50%,185.0,35.000000
75%,265.5,50.000000
max,400.0,100.000000


## 6) Dubbele `Materiaal_id`'s corrigeren
Een dubbel id krijgt het eerstvolgende vrije nummer.

In [8]:
print("Dubbele Materiaal_id's vóór correctie:", df["Materiaal_id"].duplicated().sum())

# dubbele Materiaal_id's automatisch corrigeren
duplicaten = df["Materiaal_id"].duplicated(keep="first")
for index in df.index[duplicaten]:
    huidig_id = df.loc[index, "Materiaal_id"]
    nummer = int(huidig_id.replace("LM-", ""))

    while f"LM-{nummer + 1:04d}" in df["Materiaal_id"].values:
        nummer += 1

    df.loc[index, "Materiaal_id"] = f"LM-{nummer + 1:04d}"

print("Dubbele Materiaal_id's na correctie:", df["Materiaal_id"].duplicated().sum())

Dubbele Materiaal_id's vóór correctie: 4
Dubbele Materiaal_id's na correctie: 0


## 7) Doelgroep combineren en overbodige kolommen verwijderen

In [9]:
# doelgroep en doelgroep_extra combineren en doelgroep_extra verwijderen
def combineer_doelgroep(rij):
    waarden = set()

    for kolom in ["Doelgroep", "Doelgroep_extra"]:
        waarde = rij[kolom]

        if pd.notna(waarde):
            waarden.update(waarde.lower().split("/"))

    volgorde = ["mbo", "nt2"]
    resultaat = [waarde for waarde in volgorde if waarde in waarden]

    return "/".join(resultaat) if resultaat else pd.NA
df["Doelgroep"] = df.apply(combineer_doelgroep, axis=1)
df = df.drop(columns=['Doelgroep_extra'])

# Cataloguscode verwijderen, want dit is hetzelfde als Materiaal_id
#df = df.drop(columns=['Cataloguscode'])

# Genre_kopie verwijderen, want deze bevat geen aanvullende informatie op Genre
df = df.drop(columns=['Genre_kopie'])

df["Doelgroep"].value_counts(dropna=False)

Doelgroep
mbo/nt2    212
mbo         65
NaN          3
nt2          3
Name: count, dtype: int64

## 8) Gegevensstatus en publicatiejaar corrigeren

In [10]:
# Gegevensstatus aangepast: nu is het de status beschikbaar, niet beschikbaar en uitgeleend
df["Gegevensstatus"] = df["Gegevensstatus"].replace({
    "Aangevuld (deels gesimuleerd)": "beschikbaar",
    "gecontroleerd": "niet beschikbaar",
    "In behandeling": "uitgeleend"
}).fillna("niet beschikbaar")

# twee gecontroleerde publicatiejaren corrigeren
df.loc[df["Titel"] == "De roadtrip", "Publicatiejaar"] = "2025"
df.loc[df["Titel"] == "Mijn naam is Edino", "Publicatiejaar"] = "2025"
# Overige ongeldige jaren als NaN
df["Publicatiejaar"] = df["Publicatiejaar"].replace({
    "2099": pd.NA,
    "onbekend": pd.NA,
    "nan": pd.NA
})

print(df["Gegevensstatus"].value_counts(dropna=False))
print(df["Publicatiejaar"].value_counts(dropna=False))

Gegevensstatus
beschikbaar         245
niet beschikbaar     27
uitgeleend           11
Name: count, dtype: int64
Publicatiejaar
NaN     71
2023    44
2013    42
2025    33
2018    30
1998    26
2005    21
2021    16
Name: count, dtype: int64


## 9) Type materiaal en genre opschonen
Bij waarden als `boek - thriller` of `boek - roman` wordt het type `boek` en komt `thriller` / `roman` bij het genre.

In [11]:
# Type_materiaal opschonen: # Titels met boek - thriller of boek - roman, hier boek van maken en thriller of roman als genre toevoegen
nieuwe_genres = df["Type_materiaal"].str.split(" - ").str[1]

is_leeg = (df["Genre"].isna())

staat_al_iets = (df["Genre"].notna()) & (df["Genre"] != "onbekend")

df.loc[
    df["Type_materiaal"].str.contains("boek -", na=False) & staat_al_iets,
    "Genre",
] = (
    df["Genre"] + ";" + nieuwe_genres
)

df.loc[
    df["Type_materiaal"].str.contains("boek -", na=False) & is_leeg, "Genre"
] = nieuwe_genres

# naamgeving aanpassen en de krant
materiaal_mapping = {
    "online artikel": "artikel",
    "krant (papier)": "krant",
    "digitale krant": "krant",
    "boek - thriller": "boek",
    "boek - roman": "boek",
}
df["Type_materiaal"] = df["Type_materiaal"].replace(materiaal_mapping)

print(df["Type_materiaal"].value_counts(dropna=False))
print(df["Genre"].value_counts(dropna=False))

Type_materiaal
boek           206
artikel         34
dichtbundel     23
blogpost        15
tijdschrift      2
krant            2
NaN              1
Name: count, dtype: int64
Genre
fictie             132
informatief         45
fictie;thriller     33
fictie;roman        26
poëzie              21
NaN                 20
thriller             4
roman                2
Name: count, dtype: int64


In [12]:
# onbekende waarden van de categorische kolommen 'onbekend' maken
#categorische_kolommen = [
#    "Auteur",
#    "Uitgever",
#    "Type_materiaal",
#    "Niveau",
#    "Doelgroep",
#    "Beschikbaar_formaat",
#    "Interne_batchcode",
#    "Administratieve_prioriteit",
#    "Genre"
#]
#
#df[categorische_kolommen] = df[categorische_kolommen].fillna("onbekend")

## 10) Beschikbaar formaat en auteurs opschonen

In [13]:
# online en digitaal samenvoegen
df["Beschikbaar_formaat"] = df["Beschikbaar_formaat"].replace("digitaal", "online")
# Beschikbaar_formaat is online als er een bron is
formaat_is_onbekend = (df["Beschikbaar_formaat"] == "onbekend") | (
    df["Beschikbaar_formaat"].isna()
)

heeft_bron_url = (
    (df["Bron_url"].notna())
    & (df["Bron_url"] != "onbekend")
    & (df["Bron_url"] != "")
)

df.loc[formaat_is_onbekend & heeft_bron_url, "Beschikbaar_formaat"] = "online"

df["Beschikbaar_formaat"].value_counts(dropna=False)

Beschikbaar_formaat
online    102
papier     96
pdf        54
NaN        31
Name: count, dtype: int64

In [14]:
# meerdere auteurs weergeven met ; ertussen, en Hans een achternaam geven
df["Auteur"] = df["Auteur"].str.replace(
    "Hans en Monique Hagen",
    "Hans Hagen; Monique Hagen",
    regex=False
)

df["Auteur"] = (
    df["Auteur"]
    .str.replace(r"\s+en\s+", "; ", regex=True)
    .str.replace(r"\s*&\s*", "; ", regex=True)
    .str.replace(r"\s*/\s*", "; ", regex=True)
    .str.replace(r"\s*,\s*", "; ", regex=True)
)

df.loc[df["Auteur"].str.contains(";", na=False), "Auteur"].unique()

<StringArray>
[          'Charlotte Colman; Stefanie Van Hertem',
                                        'NOS; ANP',
 'Nina Jansen; David van Hulzen; Joost Schellevis',
                               'NOS; ING Research',
               'De Vlinderstichting; Nature Today',
                      'BoerenNatuur; Nature Today',
                       'Hans Hagen; Monique Hagen']
Length: 7, dtype: str

## 11) Resultaat controleren en opslaan

In [15]:
print("Shape:", df.shape)
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "ontbrekend": df.isna().sum(),
    "uniek": df.nunique(),
})

Shape: (283, 30)


,dtype,ontbrekend,uniek
Titel,str,0,277
Auteur,str,7,229
Korte_omschrijving,str,0,282
Type_materiaal,str,1,6
Niveau,str,6,4
Themas_boek,str,183,92
Materiaal_id,str,0,283
Genre,str,20,7
Onderwerp,str,183,46
Trefwoorden,str,183,92


In [16]:
df.to_csv("../datasets/leesmateriaal_opgeschoond.csv", index=False)
print("Opgeslagen als ../datasets/leesmateriaal_opgeschoond.csv")

Opgeslagen als ../datasets/leesmateriaal_opgeschoond.csv
